In [1]:
# In this notebook we will try to solve the prolem of overfitting by using L1 and L2 regularization technique.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso


In [2]:
df = pd.read_csv('C:\\Users\\compu\\Documents\\GitHub\\GuerraAntonia_DepositoCorso\\corso python\\mercoledì 03-12\\Melbourne_housing.csv')
print("Initial shape:", df.shape)
print(df.nunique())


Initial shape: (34857, 22)
Suburb             351
Address          34009
Rooms               12
Type                 3
Method               9
SellerG            388
Date                78
Distance           215
Postcode           211
Bedroom             15
Bathroom            11
Car                 15
Landsize          1684
BuildingArea       994
YearBuilt          160
CouncilArea         33
Latitude         13402
Longtitude       14524
Regionname           8
Propertycount      342
ParkingArea          8
Price             2871
dtype: int64


C:\Users\compu\AppData\Local\Temp\ipykernel_10892\4243846038.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:\\Users\\compu\\Documents\\GitHub\\GuerraAntonia_DepositoCorso\\corso python\\mercoledì 03-12\\Melbourne_housing.csv')


In [3]:
# so we have several columns with NaN values so we need to handle these columns. We can actually fill some of these column's NaN 
# values just by 0 and some other columns might need some other treatment based on their nature for example price.
# lets first handle the columns where we need to fill only 0.
columns_to_use = ['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname', 'Propertycount', 'Distance', 'CouncilArea', 'Bedroom', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'Price']
df_new = df[columns_to_use].copy()
df_new.head()

,Suburb,Rooms,Type,Method,SellerG,Regionname,Propertycount,Distance,CouncilArea,Bedroom,Bathroom,Car,Landsize,BuildingArea,Price
0,Abbotsford,2,h,SS,Jellis,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,1.0,126.0,inf,NaN
1,Airport West,3,t,PI,Nelson,Western Metropolitan,3464.0,13.5,Moonee Valley City Council,3.0,2.0,1.0,303.0,225,840000.0
2,Albert Park,2,h,S,hockingstuart,Southern Metropolitan,3280.0,3.3,Port Phillip City Council,2.0,1.0,0.0,120.0,82,1275000.0
3,Albert Park,2,h,S,Thomson,Southern Metropolitan,3280.0,3.3,Port Phillip City Council,2.0,1.0,0.0,159.0,inf,1455000.0
4,Alphington,3,h,SN,McGrath,Northern Metropolitan,2211.0,6.4,Darebin City Council,3.0,2.0,1.0,174.0,122,NaN


In [4]:
# Now lets fill the columns named landsize and building area with mean of the whole respective columns
df_new['Landsize'] = pd.to_numeric(df_new['Landsize'], errors='coerce') # the numbers were in string form so had to convert them to integers.
df_new['BuildingArea'] = pd.to_numeric(df_new['BuildingArea'], errors='coerce')
df_new['Landsize'] = df_new['Landsize'].fillna(df_new.Landsize.mean())
df_new['BuildingArea'] = df_new['BuildingArea'].fillna(df_new.BuildingArea.mean())
df_new = df_new.replace([np.inf, -np.inf], np.nan).dropna()
print("Shape dopo i valori nulli:\n", df_new.isna().sum())



Shape dopo i valori nulli:
 Suburb           0
Rooms            0
Type             0
Method           0
SellerG          0
Regionname       0
Propertycount    0
Distance         0
CouncilArea      0
Bedroom          0
Bathroom         0
Car              0
Landsize         0
BuildingArea     0
Price            0
dtype: int64


In [5]:
# Rimozione valori infiniti
df = df.replace([np.inf, -np.inf], np.nan)
    
# Rimozione righe con Price mancante (target)
df = df.dropna(subset=['Price'])

print("Shape dopo la rimozione dei valori infiniti:\n", df_new.isna().sum())
print("Shape finale:", df_new.shape)

Shape dopo la rimozione dei valori infiniti:
 Suburb           0
Rooms            0
Type             0
Method           0
SellerG          0
Regionname       0
Propertycount    0
Distance         0
CouncilArea      0
Bedroom          0
Bathroom         0
Car              0
Landsize         0
BuildingArea     0
Price            0
dtype: int64
Shape finale: (10479, 15)


Le dummy variables servono a:

-Convertire categorie in numeri

-Evitare interpretazioni errate

-Permettere ai modelli di leggere correttamente variabili non numeriche

In [6]:
# now we are good to go with out cleaned data. Now we are going to make dummy variables for our whole dataset.
#df_new = pd.get_dummies(df_new, drop_first=True) # it is a short cut to avoid dummy variable trap it is just dropping the main column whose dummies we have produced. 
#df_new

In [7]:
# Imputazione con media per landsize e buildingarea
df['Landsize'] = pd.to_numeric(df['Landsize'], errors='coerce')
df['BuildingArea'] = pd.to_numeric(df['BuildingArea'], errors='coerce')
df['Landsize'] = df['Landsize'].fillna(df['Landsize'].median())
df['BuildingArea'] = df['BuildingArea'].fillna(df['BuildingArea'].median())

# stampa media, mediana, moda della colonna Price
col = df_new["Price"]
print("Media   :", col.mean())
print("Mediana :", col.median())
print("Moda    :", col.mode().values)



Media   : 1084781.3116709609
Mediana : 890000.0
Moda    : [600000.]


In [8]:
from sklearn.model_selection import train_test_split
x = df_new.drop('Price', axis='columns')
y = df_new.Price


x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, random_state=2)
print("Shape of X:", x.shape)
print("Shape of y:", y.shape)
print("Train set X:", x_train.shape)
print("Train set y:", y_train.shape)
print("Test set X:", x_test.shape)
print("Test set y:", y_test.shape)

Shape of X: (10479, 14)
Shape of y: (10479,)
Train set X: (8383, 14)
Train set y: (8383,)
Test set X: (2096, 14)
Test set y: (2096,)


In [9]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train, y_train)

ValueError: could not convert string to float: 'Gladstone Park'

In [ ]:
# so we can see that our model is facing the problem of overfitting because on training dataset it scores higher and on the
# testing dataset it score lower. In simple words our model is overfit to the training dataset and underfit to the testing dataset.
# We can solve the problem of overfitting by using L1 0r L2 regularization.  
from sklearn.linear_model import Lasso    # Sklearn's Lass regression is the L1 regularization. 
lasso_model = Lasso()
lasso_model.fit(x_train, y_train)
# the L1 regularization or the Lasso model will add an absolute θ value in the mean squared error

c:\Users\compu\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.300e+14, tolerance: 3.809e+11
  model = cd_fast.enet_coordinate_descent(


,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [ ]:
lasso_model.score(x_test, y_test)
# We can see that from -48 percent score to 70 percent score our model is much bette now after using L1 regularization.

0.7240356764797222

In [ ]:
lasso_model.score(x_train, y_train)

0.722210577174854

In [ ]:
# Now we will use the L2 regularization tehnique
from sklearn.linear_model import Ridge
ridge_model = Ridge(alpha=50, max_iter=100, tol=0.1)
ridge_model.fit(x_train, y_train)

,alpha,50
,fit_intercept,True
,copy_X,True
,max_iter,100
,tol,0.1
,solver,'auto'
,positive,False
,random_state,None


In [ ]:
ridge_model.score(x_test,y_test)
# after using L2 regularization our model is also much better but it seems that L1 regularization is slightly better then L2 in this case.

0.7011450723720148

In [ ]:
ridge_model.score(x_train,y_train)

0.6853723808022057

In [ ]:
#calcolo del r2
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
y_pred_lasso = lasso_model.predict(x_test)
r2_lasso = r2_score(y_test, y_pred_lasso)
print("R2 score Lasso:", r2_lasso) 
#rmse

rmse_lasso = root_mean_squared_error(y_test, y_pred_lasso)
print("RMSE Lasso:", rmse_lasso)


R2 score Lasso: 0.7240356764797222
RMSE Lasso: 368546.65981750254


In [ ]:
#ridge r quadrato
y_pred_ridge = ridge_model.predict(x_test)
r2_ridge = r2_score(y_test, y_pred_ridge)
print("R2 score Ridge:", r2_ridge)

R2 score Ridge: 0.7011450723720148


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

In [ ]:
# --- 2. Configurazione del Modello ---
# Usiamo LogisticRegression (già nota)
model = LogisticRegression(solver='liblinear') # liblinear va bene per dataset piccoli

# --- 3. Stratified K-Fold ---
# Vogliamo 5 round di validazione.
# Shuffle=True è fondamentale per mescolare i dati prima di tagliare.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- 4. Esecuzione della Cross-Validation ---
# scoring='f1': Usiamo F1-score perché l'accuratezza è inutile su dati sbilanciati
scores = cross_val_score(model, X, y, cv=cv, scoring='f1')

print("\n--- Risultati Cross-Validation (5 Folds) ---")
for i, score in enumerate(scores):
    print(f"Fold {i+1}: F1-Score = {score:.4f}")

print(f"\n>> Performance Media: {scores.mean():.4f}")
print(f">> Stabilità (Deviazione Std): +/- {scores.std():.4f}")

In [ ]:
from sklearn.linear_model import LinearRegression,Lasso, Ridge


from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score, KFold



model = Ridge(alpha=1, random_state=42)

# --- 3.K-Fold ---
# Vogliamo 5 round di validazione.
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# --- 4. Esecuzione della Cross-Validation ---
scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_squared_error')

# --- Conversione in Positivo e Calcolo RMSE ---
mse_scores = -scores # Togliamo il segno meno
rmse_scores = np.sqrt(mse_scores) # Facciamo la radice quadrata per avere l'errore

print("\n--- Risultati Cross-Validation ---")
for i, mse in enumerate(mse_scores):
    print(f"Fold {i+1}: MSE = {mse:,.0f} | RMSE = {np.sqrt(mse):,.0f}")

    print("-" * 40)
    print(f"MSE Medio: {mse_scores.mean():,.0f}")
    print(f"RMSE Medio: {rmse_scores.mean():,.0f}")
    print(f"Stabilità (Std RMSE): +/- {rmse_scores.std():,.0f}")